# Findie: Ollama và tìm đồ thất lạc bằng mô tả tiếng Việt

Notebook độc lập dành cho **thử nghiệm tương tác** trên Colab GPU. Chạy từng cell từ trên xuống; chọn **Runtime → Change runtime type → T4 GPU** nếu có. Ollama chỉ nghe trên `127.0.0.1`; không mở tunnel, nhận credential hoặc kết nối DB/RabbitMQ thật.

Mẫu gồm ví đen tại **phòng E.205**, ví đen tại **phòng E.206**, và ví đen tại **sảnh hội trường A**, cùng khu vực Trường Đại học Sài Gòn. Có thể sửa mô tả truy vấn để kiểm khả năng phân biệt địa điểm nhỏ. Đây là kiểm tra embedding văn bản; dịch vụ Python còn kết hợp từ khóa và bộ lọc. Không dùng mẫu nhỏ này để kết luận chất lượng tìm kiếm thực tế.

**Giới hạn:** Colab không bảo đảm GPU, thời lượng hoặc uptime. Không dùng notebook làm máy chủ ứng dụng thường trực hay thêm cơ chế tránh ngắt phiên. [Colab FAQ](https://research.google.com/colaboratory/faq.html) mô tả hạn chế web service/remote proxy; nếu cần endpoint cho ứng dụng, dùng hạ tầng được phép và do bạn kiểm soát. Xóa output trước khi chia sẻ notebook.


In [ ]:
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import subprocess
import time
import urllib.error
import urllib.request

# Đổi model cùng revision và dimensions; không đổi riêng tên rồi dùng lại vector cũ.
MODEL = "qwen3-embedding:4b"
EXPECTED_REVISION = "df5bd2e3c74cd8d069d21dc038f1b359fcdc9458fce1c99bd43c9eb1518ff907"
DIMENSIONS = 2048
QUERY_INSTRUCTION = "Given a Vietnamese lost-item description, retrieve matching found-item posts, preserving object details and precise finding locations."
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_VERSION = "0.40.11"
INSTALLER_SHA256 = "25f64b810b947145095956533e1bdf56eacea2673c55a7e586be4515fc882c9f"
DOWNLOAD_RESULTS = False
WORK = Path("/content/findie-ollama")
WORK.mkdir(parents=True, exist_ok=True)

if not re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9._:/-]{0,199}", MODEL):
    raise ValueError("MODEL không hợp lệ.")
if not isinstance(DIMENSIONS, int) or isinstance(DIMENSIONS, bool) or not 1 <= DIMENSIONS <= 16000:
    raise ValueError("DIMENSIONS phải từ 1 đến 16000 và được model hỗ trợ (4B: 32–2560).")
if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_REVISION):
    raise ValueError("EXPECTED_REVISION phải là digest SHA-256 đầy đủ đã kiểm tra.")


## Cài Ollama đã ghim và kiểm GPU

Cell này tải script cài đặt từ release chính thức, kiểm SHA-256 rồi cài bản `0.40.11`. Không cài Torch/Transformers. Model 4B Q4_K_M khoảng 2.5 GB trên đĩa; VRAM thực tế còn phụ thuộc context/runtime, phải đo bằng kết quả của phiên đang chạy. Cần runtime Linux GPU sạch. Thay phiên bản Ollama thì cập nhật cả checksum từ [release chính thức](https://github.com/ollama/ollama/releases/tag/v0.40.11).


In [ ]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Chọn runtime GPU rồi chạy lại notebook.")
subprocess.run(["nvidia-smi"], check=True)
if not shutil.which("zstd"):
    apt = ["apt-get"] if os.geteuid() == 0 else ["sudo", "apt-get"]
    subprocess.run([*apt, "update"], check=True)
    subprocess.run([*apt, "install", "-y", "zstd"], check=True)
installer_url = f"https://github.com/ollama/ollama/releases/download/v{OLLAMA_VERSION}/install.sh"
with urllib.request.urlopen(installer_url, timeout=60) as response:
    installer = response.read(100_000)
if hashlib.sha256(installer).hexdigest() != INSTALLER_SHA256:
    raise RuntimeError("Checksum installer không khớp; dừng trước khi thực thi.")
installer_path = WORK / "install.sh"
installer_path.write_bytes(installer)
ollama_env = {
    **os.environ,
    "OLLAMA_VERSION": OLLAMA_VERSION,
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_NO_CLOUD": "1",
    "OLLAMA_MODELS": str(WORK / "models"),
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
}
try:
    subprocess.run(["sh", str(installer_path)], env=ollama_env, check=True,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(f"Cài Ollama thất bại (exit {error.returncode}):\n{error.stdout[-8000:]}") from None


In [ ]:
def ollama_api(path, payload=None, timeout=180):
    data = None if payload is None else json.dumps(payload, allow_nan=False).encode("utf-8")
    request = urllib.request.Request(
        OLLAMA_BASE_URL + path, data=data,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)


if "ollama_process" not in globals() or ollama_process.poll() is not None:
    ollama_log = (WORK / "ollama.log").open("a", encoding="utf-8")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"], env=ollama_env, stdout=ollama_log, stderr=subprocess.STDOUT,
    )
for attempt in range(30):
    if ollama_process.poll() is not None:
        raise RuntimeError("Ollama không khởi động. Xem /content/findie-ollama/ollama.log.")
    try:
        runtime_version = ollama_api("/api/version", timeout=2)["version"]
        break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    raise RuntimeError("Ollama chưa sẵn sàng sau 30 giây.")
if runtime_version != OLLAMA_VERSION:
    raise RuntimeError(f"Ollama đang chạy {runtime_version}, cần {OLLAMA_VERSION}; dùng runtime sạch.")
print("Ollama", runtime_version, "chỉ dùng tại", OLLAMA_BASE_URL)


## Pull model và kiểm đúng revision

`model:tag` có thể đổi nội dung. Cell luôn in digest quan sát từ `/api/tags`, rồi so sánh với `EXPECTED_REVISION` trước khi tạo vector. Nếu đổi model, kiểm model card/digest chính thức rồi sửa cấu hình và chạy lại; không bỏ kiểm tra này. `qwen3-embedding:4b` là model **text-only**, khác model Qwen3-VL trong notebook 01.


In [ ]:
subprocess.run(["ollama", "pull", MODEL], env=ollama_env, check=True)
models = ollama_api("/api/tags")["models"]
model_info = next((entry for entry in models if entry.get("name") == MODEL), None)
if model_info is None:
    raise RuntimeError("Không tìm thấy model đúng tên trong /api/tags.")
observed_revision = model_info["digest"].removeprefix("sha256:")
print("Model:", model_info["name"])
print("AI_EMBEDDING_REVISION=" + observed_revision)
print("Details:", model_info.get("details", {}))
if observed_revision != EXPECTED_REVISION:
    raise RuntimeError("Digest không khớp. Xác minh model mới trước khi cập nhật revision/reindex.")


In [ ]:
def validate_vector(vector, dimensions):
    if not isinstance(vector, list) or len(vector) != dimensions:
        raise ValueError("Vector sai số chiều.")
    if any(isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) for value in vector):
        raise ValueError("Vector phải chứa số hữu hạn.")
    norm = math.hypot(*vector)
    if not math.isfinite(norm) or norm == 0:
        raise ValueError("Vector không thể chuẩn hóa L2.")
    return [value / norm for value in vector]


def embed(texts, *, query=False):
    if not isinstance(texts, list) or not texts or any(not isinstance(text, str) or not text.strip() for text in texts):
        raise ValueError("Cần danh sách mô tả không rỗng.")
    if query and QUERY_INSTRUCTION:
        texts = [f"Instruct: {QUERY_INSTRUCTION}\nQuery: {text}" for text in texts]
    payload = ollama_api("/api/embed", {
        "model": MODEL, "input": texts, "dimensions": DIMENSIONS, "truncate": False,
    })
    if payload.get("model") != MODEL:
        raise ValueError("Ollama trả sai tên model.")
    vectors = payload.get("embeddings")
    if not isinstance(vectors, list) or len(vectors) != len(texts):
        raise ValueError("Ollama trả sai số lượng vector.")
    return [validate_vector(vector, DIMENSIONS) for vector in vectors]


def rank_documents(query_vector, document_vectors, records):
    if len(document_vectors) != len(records):
        raise ValueError("Số vector không khớp số món đồ.")
    query_vector = validate_vector(query_vector, len(query_vector))
    scores = []
    for record, vector in zip(records, document_vectors):
        vector = validate_vector(vector, len(query_vector))
        scores.append({"id": record["id"], "description": record["description"],
                       "cosine": math.fsum(q * d for q, d in zip(query_vector, vector))})
    return sorted(scores, key=lambda row: (-row["cosine"], row["id"]))


assert validate_vector([3, 4], 2) == [0.6, 0.8]
assert rank_documents([1, 0], [[0, 1], [1, 0]], [
    {"id": "e206", "description": "E.206"}, {"id": "e205", "description": "E.205"},
])[0]["id"] == "e205"
print("Các kiểm tra vector và xếp hạng cơ bản đã qua.")


## Dữ liệu giả lập và truy vấn có thể sửa

Khu vực đã chọn là **Trường Đại học Sài Gòn**; tên phòng/sảnh nằm nguyên trong mô tả. Notebook chỉ giữ một khu vực để quan sát ảnh hưởng của mô tả chi tiết. Dịch vụ thực tế phải giữ bộ lọc khu vực riêng, không nhờ model suy ra hoặc tự nới khu vực.

Sửa `QUERY` rồi chạy lại cell truy vấn. Điểm cosine thể hiện độ gần vector, không phải xác suất đúng chủ; các phòng chỉ khác một chữ số có thể rất gần nhau.


In [ ]:
AREA_NAME = "Trường Đại học Sài Gòn"
RECORDS = [
    {"id": "wallet-e205", "description": "Nhặt được ví da màu đen tại phòng học E.205, Trường Đại học Sài Gòn."},
    {"id": "wallet-e206", "description": "Nhặt được ví da màu đen tại phòng học E.206, Trường Đại học Sài Gòn."},
    {"id": "wallet-hall-a", "description": "Nhặt được ví da màu đen tại sảnh hội trường A, Trường Đại học Sài Gòn."},
    {"id": "bottle-e205", "description": "Nhặt được bình nước màu xanh tại phòng học E.205, Trường Đại học Sài Gòn."},
]
document_vectors = embed([record["description"] for record in RECORDS])
print(f"Đã kiểm {len(document_vectors)} vector, mỗi vector {DIMENSIONS} chiều, hữu hạn và L2.")
subprocess.run(["ollama", "ps"], env=ollama_env, check=True)


In [ ]:
QUERY = "Tôi mất ví da đen ở phòng E.205 trường Đại học Sài Gòn"
query_vector = embed([QUERY], query=True)[0]
ranking = rank_documents(query_vector, document_vectors, RECORDS)
for rank, result in enumerate(ranking, 1):
    print(rank, result["id"], f"cosine={result['cosine']:.6f}", result["description"])


In [ ]:
SAMPLE_CASES = [
    {"query": "Mất ví da đen tại phòng E.205 Đại học Sài Gòn", "expected_id": "wallet-e205"},
    {"query": "Mat vi da den tai phong E.206 Dai hoc Sai Gon", "expected_id": "wallet-e206"},
    {"query": "Tìm ví đen đánh rơi ở sảnh hội trường A Đại học Sài Gòn", "expected_id": "wallet-hall-a"},
]
sample_evaluation = []
for case in SAMPLE_CASES:
    vector = embed([case["query"]], query=True)[0]
    results = rank_documents(vector, document_vectors, RECORDS)
    sample_evaluation.append({**case, "ranking": results, "hit_at_1": results[0]["id"] == case["expected_id"]})
for case in sample_evaluation:
    print(case["expected_id"], "Hit@1:", case["hit_at_1"], "top:", case["ranking"][0]["id"])
print("Mẫu giả lập chỉ để kiểm luồng; cần dữ liệu có nhãn độc lập để đánh giá độ chính xác.")


## Xuất cấu hình và kết quả mẫu

Chỉ xuất model/digest/số chiều/prompt, thông tin runtime và ba ca mẫu; không xuất vector, câu truy vấn tùy ý ở cell `QUERY`, `.env`, khóa API hay credential. File không tự import vào DB. Cấu hình ở dưới là danh tính model, chưa cấu hình kết nối dịch vụ. Đặt `DOWNLOAD_RESULTS=True` ở cell đầu nếu muốn Colab tải JSON về.


In [ ]:
export = {
    "kind": "findie-ollama-interactive-sample",
    "service_model_config": {
        "AI_EMBEDDING_MODEL": MODEL,
        "AI_EMBEDDING_REVISION": observed_revision,
        "AI_EMBEDDING_DIMENSIONS": DIMENSIONS,
        "AI_EMBEDDING_QUERY_INSTRUCTION": QUERY_INSTRUCTION,
    },
    "ollama_version": runtime_version,
    "query_instruction": QUERY_INSTRUCTION,
    "dimensions": DIMENSIONS,
    "evaluation": "synthetic_dense_only_not_a_production_quality_benchmark",
    "sample_cases": sample_evaluation,
}
export_path = WORK / "findie-ollama-sample.json"
export_path.write_text(json.dumps(export, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
print("Kết quả mẫu:", export_path)
print(json.dumps(export["service_model_config"], indent=2))
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(export_path))


## Kết thúc phiên

Chạy cell dưới để dừng tiến trình do notebook tạo, rồi **Runtime → Disconnect and delete runtime**. Không giữ phiên bằng tác vụ nền. Xóa mọi output nếu lưu/chia sẻ notebook.


In [ ]:
if "ollama_process" in globals() and ollama_process.poll() is None:
    ollama_process.terminate()
    try:
        ollama_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        ollama_process.kill()
        ollama_process.wait(timeout=10)
if "ollama_log" in globals():
    ollama_log.close()
print("Đã dừng Ollama của notebook; có thể xóa runtime.")


## Nguồn chính thức

- [Ollama Linux installation](https://docs.ollama.com/linux), [release v0.40.11](https://github.com/ollama/ollama/releases/tag/v0.40.11).
- [Qwen3-Embedding:4b trên Ollama](https://ollama.com/library/qwen3-embedding:4b), [model card Qwen3-Embedding-4B](https://huggingface.co/Qwen/Qwen3-Embedding-4B).
- [Ollama `/api/embed`](https://docs.ollama.com/api/embed), [`/api/tags`](https://docs.ollama.com/api/tags), [authentication](https://docs.ollama.com/api/authentication).
- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html).

Đối chiếu ngày 25/10/2026; chưa chạy notebook trên Colab GPU thật trong môi trường phát triển.
